# CFDE / MoTrPAC / GTEx Orientation

This is one chapter of the *From Sequence to Variant: Protein Language Models on CFDE Data* training module. Follow the numbered chapters in order.


---
## Part 0: Orientation (10 min)

### What is CFDE, and why does cross-program data integration matter?

The **Common Fund Data Ecosystem (CFDE)** connects data generated across NIH Common Fund programs, each run by an independent Data Coordinating Center (DCC), each with its own data types, formats, and access tools. Individually, these are powerful single-purpose resources. The scientific opportunity, and the harder technical problem, is *linking* them: asking a question that only makes sense when you combine information that lives in two different programs.

That is exactly what this module does: MoTrPAC tells you *which proteins respond to exercise*; GTEx tells you *what regulatory DNA variants control those same genes' expression*. Neither dataset alone answers the sequence-to-variant question we're asking today.

### MoTrPAC in brief

The **Molecular Transducers of Physical Activity Consortium (MoTrPAC)** measures multi-omic (transcriptomic, proteomic, and more) changes across many tissues in response to structured endurance exercise training. The dataset we'll use is the **rat 6-month endurance training study** — fully open, no registration required, licensed CC BY 4.0. "Exercise-responsive" in this context means a gene or protein shows a statistically significant expression change between exercise-trained and sedentary control animals in a given tissue.

### GTEx in brief

The **Genotype-Tissue Expression (GTEx)** project characterizes how genetic variation affects gene expression across dozens of human tissues. An **expression quantitative trait locus (eQTL)** is a genomic position where a DNA variant is statistically associated with the expression level of a nearby (or distant) gene. eQTLs are one of the primary tools researchers use to interpret what non-coding genetic variants might *do*.

### Why a protein language model bridges them

MoTrPAC gives us *labels* (exercise-responsive or not). GTEx gives us *variant positions*. What's missing is a way to ask whether something about the **protein sequence itself** predicts exercise-responsiveness, and if so, whether the sequence positions that matter to the model line up with the regulatory variant positions GTEx has already found. ESM2, a protein language model pre-trained on millions of sequences, gives us both a predictive model and an interpretability lens to ask exactly that.


### 🧠 Concept Check
1. In your own words, why isn't MoTrPAC data alone enough to ask about genetic variants?
2. Why isn't GTEx data alone enough to ask about exercise-responsiveness?


In [1]:
# ============================================================
# CELL PURPOSE: set up the Python environment. Run this cell first, every time.
# ============================================================

import sys
# This checks whether we're running inside Google Colab (vs. a local computer).
# "google.colab" is a module that only exists inside Colab notebooks.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # "!pip install" runs a command-line install of the Python packages this notebook needs.
    # -q means "quiet" (less scrolling text). You only need to do this once per Colab session.
    # pyreadr    -> lets Python read R's .rda data files (MoTrPAC's data format) without needing R
    # requests   -> lets Python make web requests (download files, call APIs)
    # biopython  -> biology-specific tools; we use its sequence-alignment feature in Part 4
    # fair-esm   -> Meta AI's protein language model (ESM2) that we fine-tune in Part 3
    # torch      -> PyTorch, the deep learning library ESM2 is built on
    # scikit-learn -> standard machine learning library (our classifier lives here)
    # matplotlib -> for making plots
    # pandas / numpy -> standard data-handling libraries (tables and math)
    !pip install -q pyreadr requests biopython fair-esm torch scikit-learn matplotlib pandas numpy

# "import" loads a library so we can use it later in the notebook.
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Environment ready. IN_COLAB =", IN_COLAB)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.4/788.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.8 MB/s eta 0:00:00
Environment ready. IN_COLAB = True
